In [0]:
%sql
use catalog demo_workspace

In [0]:
%run /Workspace/Users/erica.roque@gmail.com/Databricks-Certified-Data-Engineer-Associate/Includes/Copy-Datasets

In [0]:
%sql
use catalog demo_workspace;
use schema book_store

In [0]:
%sql
select * from customers

In [0]:
customers_df =  spark.read.table("demo_workspace.book_store.customers")
customers_df.show()

In [0]:
display(customers_df.select("customer_id","email"))

In [0]:
from pyspark.sql.functions import col

display(customers_df.select(col("customer_id"),col("email").alias("customer_email"),col("updated").cast("timestamp")))

In [0]:
display(customers_df.drop("updated"))

In [0]:
from pyspark.sql.functions import col, split

display(customers_df.withColumn("domain",split(col("email"),"@").getItem(1)))

In [0]:
new_customer_df =customers_df.withColumnRenamed("email","customer_email")
display(new_customer_df)

In [0]:
from pyspark.sql.functions import col, split

new_customer_df = customers_df.withColumn("customer_domain",split(col("email"),"@").getItem(1)) \
                              .withColumnRenamed("email", "customer_email") \
                              .drop("profile", "updated")


display(new_customer_df)

In [0]:
from pyspark.sql import functions as f

new_customer_df = customers_df.withColumn("customer_domain",f.split(f.col("email"),"@").getItem(1)) \
                              .withColumnRenamed("email", "customer_email") \
                              .drop("profile", "updated")


display(new_customer_df)

In [0]:
orders_df = spark.table("orders")

#display(orders_df.describe())
display(orders_df.summary()) 
                        

In [0]:
display(orders_df.count())

In [0]:

deduped_dp = orders_df.dropDuplicates()
display(deduped_dp.count()) 


In [0]:
deduped_dp = orders_df.dropDuplicates()
display(deduped_dp.count()) 

In [0]:
clean_df = customers_df.fillna({"email":"unknown"})
display(clean_df)


In [0]:
joing_df =  customers_df.join(orders_df,"customer_id", "inner")
display(joing_df)

In [0]:
joing_df =  customers_df.join(orders_df,"customer_id", "left_anti")
display(joing_df)

In [0]:
display(orders_df.where("customer_id = 'C01194'"))

In [0]:
books_df = spark.table("books")
croos_join_df = customers_df.crossJoin(books_df)
display(croos_join_df.orderBy("customer_id"))

In [0]:
print(f"customer count: {customers_df.count()}")
print(f"books count: {books_df.count()}")
print(f"customer-book pair: {croos_join_df.count()}")

In [0]:
from pyspark.sql.functions import explode, broadcast

explode_orders_df = (
    orders_df
    .withColumn("book", explode("books"))
    .select("*", "book.book_id", "book.subtotal")
    .drop("books", "book")
)

orders_details_df = explode_orders_df.join(books_df, "book_id", "inner")

display(orders_details_df.explain())

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 20 * 1024 * 1024)

In [0]:
spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
# nao é suportado na serverless


In [0]:
customers_detail_df = spark.sql("""
                                select customer_id, email, updated,
                                profile:first_name as first_name,
                                profile:gender as gender,
                                profile:address:city as city,       
                                profile:last_name as last_name,
                                profile:address:country as country,
                                profile:address:state as state
                               from customers                              
                                """)
display(customers_detail_df)

In [0]:
customers_detail_df.write.mode("overwrite").saveAsTable("customers_detail")